In [ ]:
import requests, xml.etree.ElementTree as ET, json

sitemap_urls = [
    'https://hoatuoimymy.com/product-sitemap0.xml',
    'https://hoatuoimymy.com/product-sitemap1.xml',
    'https://hoatuoimymy.com/product-sitemap2.xml',
    'https://hoatuoimymy.com/product-sitemap3.xml'
]

all_urls = []
for url in sitemap_urls:
    try:
        r = requests.get(url)
        r.raise_for_status()
        root = ET.fromstring(r.content)
        all_urls += [loc.text for loc in root.iter('{http://www.sitemaps.org/schemas/sitemap/0.9}loc')]
    except Exception as e:
        print(f"Lỗi với {url}: {e}")

with open('all_urls.json', 'w') as f:
    json.dump(all_urls, f, indent=4)

print(f"Extracted  {len(all_urls)} URLs and saved to all_urls.json")


Danh sách các trang web

In [ ]:
import requests
from bs4 import BeautifulSoup
import json
import pandas as pd
from tqdm.notebook import tqdm

with open('all_urls.json', 'r', encoding='utf-8') as f:
    urls = json.load(f)

data = []

for url in tqdm(urls):
    try:
        resp = requests.get(url, timeout=10)
        soup = BeautifulSoup(resp.content, 'html.parser')
        
        # Lấy title, price như cũ
        title = soup.select_one('h1.product_title')
        price = soup.select_one('.price span.amount')
        
        # Lấy mô tả sản phẩm (description) - lấy cả nhiều nguồn
        desc = None
        # 1. WooCommerce short description
        desc_tag = soup.select_one('.woocommerce-product-details__short-description')
        if desc_tag:
            desc = desc_tag.get_text(separator=' ', strip=True)
        # 2. Thử lấy từ .product-short-description
        if not desc:
            desc_tag2 = soup.select_one('.product-short-description')
            if desc_tag2:
                desc = desc_tag2.get_text(separator=' ', strip=True)
        # 3. Thử lấy từ meta description
        if not desc:
            meta_desc = soup.find('meta', attrs={'name': 'description'})
            if meta_desc and meta_desc.has_attr('content'):
                desc = meta_desc['content']
        
        # Lấy thông tin khuyến mãi (nếu có)
        khuyen_mai = None
        km_tag = soup.select_one('div.khuyen-mai')
        if km_tag:
            # Lấy text của các <li> trong khuyến mãi
            km_items = km_tag.find_all('li')
            if km_items:
                khuyen_mai = "; ".join([li.get_text(separator=' ', strip=True) for li in km_items])
            else:
                khuyen_mai = km_tag.get_text(separator=' ', strip=True)
        
        # Không gộp khuyến mãi vào description nữa, để riêng
        full_desc = desc if desc else None
        
        # Lấy ảnh: thử nhiều selector
        image = None
        # 1. Ảnh trong gallery
        img_tag = soup.select_one('.woocommerce-product-gallery__image img')
        if img_tag and img_tag.has_attr('src'):
            image = img_tag['src']
        # 2. Ảnh đại diện sản phẩm
        if not image:
            img_tag = soup.select_one('img.wp-post-image')
            if img_tag and img_tag.has_attr('src'):
                image = img_tag['src']
        # 3. Ảnh trong meta property
        if not image:
            meta_img = soup.find('meta', property='og:image')
            if meta_img and meta_img.has_attr('content'):
                image = meta_img['content']
        
        data.append({
            'url': url,
            'title': title.get_text(strip=True) if title else None,
            'price': price.get_text(strip=True) if price else None,
            'description': full_desc,
            'khuyen_mai': khuyen_mai,
            'image': image
        })
    except Exception as e:
        print(f"Lỗi với {url}: {e}")

df = pd.DataFrame(data)
df.to_csv('products.csv', index=False, encoding='utf-8-sig')
print("Đã lưu dữ liệu ra products.csv")

In [ ]:
# read_CSV
df = pd.read_csv('products.csv')
# Display the first few rows of the DataFrame   
df

In [ ]:
df.head()

In [ ]:
from dotenv import load_dotenv
load_dotenv()
import os
import sys
import google.generativeai as genai
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance



In [ ]:
url = os.getenv("QDRANT_ENDPOINT")
key = os.getenv("QDRANT_API_KEY")

client = QdrantClient(
        url=url,
        api_key=key
    )

In [ ]:
client.get_collections()

In [ ]:
# create collection 
collection_name = "RAG_Huy"
if not client.collection_exists(collection_name):
    client.create_collection(
            collection_name=collection_name,
            vectors_config=VectorParams(size=768, distance=Distance.COSINE, on_disk=True),
            shard_number=2,
            timeout=180
        )
    print(f"Collection '{collection_name}' created.")
else:
    print(f"Collection '{collection_name}' already exists.")

In [ ]:
model = genai.GenerativeModel('gemini-2.0-flash')


In [ ]:

from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer("Alibaba-NLP/gte-multilingual-base", 
                                      trust_remote_code=True)

In [ ]:
def get_vector(text):
    """
    Generate an embedding for the given text using the preloaded model.

    Args:
        text (str): The input text to encode.

    Returns:
        list: The embedding as a list of floats, or an empty list if input is empty.
    """
    if not text.strip():
        print("Attempted to get embedding for empty text.")
        return []

    embedding = embedding_model.encode(text)
    return embedding.tolist()

In [ ]:
len(get_vector("Hello world")) # Test embedding function

In [ ]:
df.head()

In [ ]:
df['content'] = df['title'] +  ' giá ' + df['price'] + ' mô tả sản phẩm: ' + df['description'] + ' khuyêt mãi: ' + df['khuyen_mai'] + ' xem ảnh tại ' + df['image'].fillna('') 
df.head()

In [ ]:
df['vector'] = df['content'].astype(str).apply(lambda x: get_vector(x) if isinstance(x, str) else [])
df.head()

In [ ]:
import qdrant_client
from qdrant_client.http import models as qdrant_models
import uuid

collection_name = "RAG_Huy"

# To avoid WriteTimeout, split the upsert into smaller batches
import math

BATCH_SIZE = 50  # You can adjust this value as needed

def url_to_uuid(url):
    """
    Convert a URL string to a UUID using uuid5 and a fixed namespace.
    This ensures the same URL always maps to the same UUID.
    """
    return str(uuid.uuid5(uuid.NAMESPACE_URL, str(url)))

points = []
for idx, row in df.iterrows():
    vector = row.get("vector")
    payload = {
        "url": row.get("url"),
        "title": row.get("title"),
        "price": row.get("price"),
        "description": row.get("description"),
        "khuyen_mai": row.get("khuyen_mai"),
        "image": row.get("image"),
        "id": int(row.get("id")) if not pd.isna(row.get("id")) else None
    }
    # Qdrant requires point id to be an unsigned integer or a UUID.
    # We'll use a UUID generated from the URL for uniqueness and compliance.
    url_val = row.get("url")
    point_id = url_to_uuid(url_val) if url_val else str(uuid.uuid4())
    points.append(
        qdrant_models.PointStruct(
            id=point_id,
            vector=vector,
            payload=payload
        )
    )

# Upsert in batches to avoid timeouts
total_points = len(points)
num_batches = math.ceil(total_points / BATCH_SIZE)

for i in range(num_batches):
    batch_points = points[i*BATCH_SIZE : (i+1)*BATCH_SIZE]
    try:
        client.upsert(
            collection_name=collection_name,
            points=batch_points,
        )
        print(f"Upserted batch {i+1}/{num_batches} ({len(batch_points)} points) to Qdrant collection '{collection_name}'.")
    except Exception as e:
        print(f"Error upserting batch {i+1}: {e}")

print(f"Finished upserting {total_points} points to Qdrant collection '{collection_name}'.")


In [ ]:
#  query
query = "hoa tươi sinh nhật"
query_vector = get_vector(query)
search_result = client.search(
    collection_name=collection_name,
    query_vector=query_vector,
    limit=5,  # Limit the number of results
    with_payload=True  # Include payload in the results
)

for record in search_result:
    print(f" Payload: {record.payload}, score: {record.score}")
    